# 🚩 Notebook 2: Feature Flags

A **feature flag** lets you toggle behaviour at runtime — *without a deploy*.

Common uses:
- Hide an unfinished feature behind `off`.
- Canary-release to 5% of users and watch metrics.
- Emergency **kill-switch** for a buggy feature.
- Per-environment toggles (e.g. `debug_mode` only in staging).


## 🛠️ Setup

```bash
cd 05-microservices/configuration-externalization
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: if-statements tied to deployment

Toggling `NEW_CHECKOUT = True` below requires a code change and a deploy —
exactly what feature flags exist to avoid.


In [ ]:
NEW_CHECKOUT = False  # flip and redeploy to enable 😬

def checkout(user_id: int):
    if NEW_CHECKOUT:
        return f"user {user_id} -> new checkout"
    return f"user {user_id} -> legacy checkout"

print(checkout(1))


## 🟩 GOOD: a tiny `FlagStore`

In production you'd use a service like **LaunchDarkly**, **Unleash**, or
**Flagsmith**. The API is roughly the same: ask *"is flag X on for user Y?"*.
Here we fake one in-memory.


In [ ]:
import hashlib

class FlagStore:
    """Pretend remote config service.

    Flag values can be:
      - bool           -> on/off for everyone
      - {"percent": N} -> stable N% rollout keyed by user_id
      - {"users": [..]} -> explicit allow-list
    """
    def __init__(self, flags: dict):
        self.flags = dict(flags)

    def update(self, new: dict):
        self.flags.update(new)  # hot reload, no restart

    def enabled(self, flag: str, user_id: int | None = None) -> bool:
        f = self.flags.get(flag)
        if f is None or f is False:
            return False
        if f is True:
            return True
        if isinstance(f, dict):
            if user_id is not None and user_id in f.get("users", []):
                return True
            if "percent" in f and user_id is not None:
                h = int(hashlib.md5(f"{flag}:{user_id}".encode()).hexdigest(), 16) % 100
                return h < f["percent"]
        return False

flags = FlagStore({
    "dark_mode": True,                 # everyone
    "new_checkout": {"percent": 20},   # 20% canary
    "experimental_search": False,      # off
    "beta_ui": {"users": [42, 99]},    # internal allow-list
})

for uid in range(6):
    print(
        f"user {uid}: dark_mode={flags.enabled('dark_mode', uid)} "
        f"new_checkout={flags.enabled('new_checkout', uid)} "
        f"beta_ui={flags.enabled('beta_ui', uid)}"
    )


### 💡 Why a hash? (stable bucketing)

Using `md5(flag:user_id) % 100` means the *same* user always lands in the
same bucket — they don't flip between "on" and "off" on every request.
That's essential for consistent UX during a canary.


## 🧯 Kill-switch without a redeploy

Ops sees a bug in `new_checkout` at 3 AM. Flip it off instantly:


In [ ]:
flags.update({"new_checkout": False})

print("after kill-switch:")
for uid in range(6):
    print(f"  user {uid}: new_checkout={flags.enabled('new_checkout', uid)}")


## 🎯 Targeting by user *attributes* (not just id)

Real flag services let you target by country, plan, device, etc.
Same idea — extend the rule language to look at a `context` dict.


In [ ]:
class AttrFlagStore:
    """Flag values can be:
      - bool
      - {"percent": N}                           -> % rollout by stable hash of user_id
      - {"users": [..]}                          -> allow-list
      - {"when": {"country": ["US", "CA"]}}       -> match any attribute value
      - combinations are OR'd together
    """
    def __init__(self, flags: dict):
        self.flags = dict(flags)

    def enabled(self, flag: str, ctx: dict) -> bool:
        f = self.flags.get(flag)
        if f is True:  return True
        if not f:      return False
        if isinstance(f, dict):
            uid = ctx.get("user_id")
            if uid is not None and uid in f.get("users", []):
                return True
            when = f.get("when", {})
            if when and all(ctx.get(k) in vs for k, vs in when.items()):
                return True
            if "percent" in f and uid is not None:
                h = int(hashlib.md5(f"{flag}:{uid}".encode()).hexdigest(), 16) % 100
                if h < f["percent"]:
                    return True
        return False

store = AttrFlagStore({
    # roll out only to paid users in the US/Canada
    "fast_checkout": {"when": {"country": ["US", "CA"], "plan": ["pro", "team"]}},
    # plus a separate 10% experiment everywhere
    "recommendations_v2": {"percent": 10},
})

users = [
    {"user_id": 1, "country": "US", "plan": "pro"},
    {"user_id": 2, "country": "US", "plan": "free"},
    {"user_id": 3, "country": "DE", "plan": "pro"},
]
for u in users:
    print(u, "=> fast_checkout=", store.enabled("fast_checkout", u),
          "recs_v2=", store.enabled("recommendations_v2", u))


## 🧪 Tiny A/B test: measure the variant, don't just toggle it

A feature flag is only half the story — you also want to know which
*variant* performed better. The pattern: pick a variant from the flag, run
the user's request, then record the outcome labelled with that variant.


In [ ]:
from collections import Counter
import random

variants = Counter()
successes = Counter()

def checkout_ab(user_id: int):
    variant = "new" if flags.enabled("new_checkout", user_id) else "legacy"
    variants[variant] += 1
    # pretend: new checkout succeeds 95% of the time, legacy 90%
    ok = random.random() < (0.95 if variant == "new" else 0.90)
    if ok:
        successes[variant] += 1
    return variant, ok

# turn the canary back on to 50% for this demo
flags.update({"new_checkout": {"percent": 50}})
random.seed(0)
for uid in range(2000):
    checkout_ab(uid)

for v in ("legacy", "new"):
    n = variants[v] or 1
    print(f"{v:6s}: n={variants[v]:>4d}  success_rate={successes[v]/n:.3f}")


## 🌍 Environment-aware flags

Typical pattern: the *same* code ships everywhere, but the flag store is
initialised differently per environment (via env vars from Notebook 1!).


In [ ]:
import os

def flags_for_env(env_name: str) -> FlagStore:
    base = {"dark_mode": True}
    if env_name == "staging":
        return FlagStore({**base, "experimental_search": True, "debug_panel": True})
    if env_name == "production":
        return FlagStore({**base, "experimental_search": {"percent": 5}})
    # dev / tests
    return FlagStore({**base, "experimental_search": True, "debug_panel": True})

for env in ("dev", "staging", "production"):
    fs = flags_for_env(env)
    print(env, "->", fs.flags)


## 🆚 Environment variables vs feature flags

| | Environment var | Feature flag |
|--|--|--|
| Change requires deploy? | yes (restart) | no (hot reload) |
| Per-user targeting? | hard | easy |
| Good for secrets? | yes | **no** |
| Good for experiments? | no | yes |
| Typical change frequency | rare | often |

**Rule of thumb:** use env vars for *how the app is wired* (DB, ports, keys)
and feature flags for *what the product does* (UX variants, experiments).


## ⚠️ Flag hygiene (real-world traps)

- **Flag debt**: dead flags accumulate forever. Set an owner + removal date.
- **Default off**: new flags should fail closed.
- **Test both states**: `on` *and* `off` need CI coverage.
- **Don't use flags for secrets** — they're usually readable by many people.
